---
### *Practical AgentOps: From PoC to Prod with MLflow 3 — ODSC AI East 2026*
---



# 2.1 — MLflow AI Gateway

---

**Scenario:** Your MLflow assistant works locally. But API keys are scattered across `.env` files, every developer has their own model config, and there's no centralized way to manage costs or switch providers.

**Objective:** Use the **MLflow AI Gateway** to centralize model access behind managed endpoints

3 Ways to call endpoints you establish in your MLflow server:

1. HTTP/Invocations API
2. Completions API (OpenAI SDK) - Limited Provider Compatibility
3. Endpoint specific SDK + Passthrough API - Full Provider Compatibility

In [ ]:
import os
from dotenv import load_dotenv
import mlflow

load_dotenv()

## Set Up via UI

1. Add API Keys under AI Gateway
2. Create Endpoints or use Passthrough APIs

## Create Fallbacks (Useful for Gemini Free Tier)

### Google/Gemini API Free Tier

| Model | Category | RPM | TPM | RPD |
|---|---|---|---|---|
| Gemini 3.1 Flash Lite (preview) | Text-out models | 0 / 15 | 135 / 250K | 6 / 500 |
| Gemini 3 Flash (preview) | Text-out models | 0 / 5 | 706 / 250K | 15 / 20 |
| Gemini 2.5 Flash | Text-out models | 0 / 5 | 0 / 250K | — |

In [14]:
GEMINI_FREE="gemini-3.1-free-tier"
GEMINI_PAID=""

In [5]:
import requests

def test_model_endpoint_http(endpoint: str, question: str) -> str:
    response = requests.post(
        f"{os.environ["MLFLOW_TRACKING_URI"]}/gateway/{endpoint}/mlflow/invocations",
        json={"messages": [{"role": "user", "content": "Hello!"}], "temperature": 0.7},
    )
    return response.json()['choices'][0]['message']['content'] 

In [ ]:
print(test_model_endpoint_http(GEMINI_FREE, "Hello"))
#test_model_endpoint_http(GEMINI_3_1, "Hello")
#test_model_endpoint_http(GEMINI_3, "Hello")

Hello! How can I help you today?
Hello! How can I help you today?


# Captured the traces without any mlflow setup

The MLflow Invocations API supports both OpenAI-style chat completions and embeddings endpoints. The request body follows the OpenAI chat completions format with these supported parameters.

![](../../assets/ai_gateway_completions.png)

# Calling with OpenAI Client

In [10]:
from openai import OpenAI

openai_client = OpenAI(
    base_url=f"{os.environ["MLFLOW_TRACKING_URI"]}/gateway/mlflow/v1",
    api_key="" # not needed, configured server-side
)

SYSTEM_PROMPT = """You are an MLflow assistant."""

# From Experiment 1
def mlflow_agent(question: str) -> str:
    response = openai_client.chat.completions.create(
        model=GEMINI_FREE,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": question},
        ],
        temperature=0,
        max_completion_tokens=512,
    )
    return response.choices[0].message.content

In [ ]:
from mlflow.genai.datasets import get_dataset
retrieved_dataset = get_dataset(name="mlflow-agent-eval")

In [11]:
eval_dataset = [
    {
        "inputs": {"question": "How do I log a metric in MLflow?"},
        "expectations": {
            "expected_facts": ["mlflow.log_metric", "key", "value"]
        },
    },
]

In [12]:
if not os.getenv("MLFLOW_TRACKING_URI"):
    mlflow.set_tracking_uri("http://127.0.0.1:5000")

mlflow.set_experiment(os.getenv("EXPERIMENT_2_NAME","mlflow-agent-2"))
mlflow.openai.autolog()

In [16]:
# Evaluate
from mlflow.genai import evaluate
from mlflow.genai.scorers import Correctness

validation_results = evaluate(
    data=eval_dataset,
    predict_fn=mlflow_agent,
    scorers=[Correctness(model=f"gateway:/{GEMINI_FREE}")],
)

2026/04/21 21:49:54 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/04/21 21:49:54 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
Evaluating: 100%|██████████| 1/1 [Elapsed: 00:20, Remaining: 00:00] [predict_fn: 46%, scorers: 54%]


> **What just happened:** Both the `predict_fn` calls *and* the judge scorer calls went through the gateway. The gateway handled routing to Gemini, managing the API key, and enforcing any rate limits. No API keys in code, no provider-specific logic.

# Passthrough APIs (Server-side API keys)

Use local clients through MLflow endpoint to access full API support.

**Must use corresponding SDK to make this work**

1. `uv add google`
2. `uv sync`

In [ ]:
from google import genai

client = genai.Client(
    api_key="dummy",
    http_options={
        "base_url": f"{os.environ["MLFLOW_TRACKING_URI"]}/gateway/gemini",
    },
)

response = client.models.generate_content(
    model=GEMINI_FREE,
    contents={"text": "Hello!"},
)
client.close()
print(response.candidates[0].content.parts[0].text)

Hello! How can I help you today?


# Learn More